[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/mlops/01-registro-modelos.ipynb)

# Registro de Modelos LLM con MLflow

> Aprende a rastrear experimentos, versionar modelos y gestionar el ciclo
> de vida de sistemas LLM usando MLflow como registro central.

## Objetivos
- Registrar experimentos LLM con parámetros, métricas y artefactos
- Versionar prompts como artefactos en MLflow
- Comparar versiones de prompts con métricas de calidad
- Implementar un sistema de staging/producción para prompts

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic mlflow pandas --quiet

## 2. Configurar MLflow y cliente Anthropic

In [ ]:
import anthropic
import mlflow
import mlflow.pyfunc
import pandas as pd
import json
import time
import hashlib
from datetime import datetime

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

# Configurar MLflow con tracking local
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("llm-prompt-engineering")

print("=== REGISTRO DE MODELOS LLM CON MLFLOW ===")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experimento activo: llm-prompt-engineering")
print("\nEstructura de MLflow para LLMs:")
print("  Experiment → agrupa runs relacionados")
print("  Run → una ejecución con parámetros + métricas + artefactos")
print("  Artefactos → prompts, ejemplos, resultados, modelos")
print("  Model Registry → versiona y gestiona el ciclo de vida")

## 3. Registrar experimentos de prompts

In [ ]:
# Dataset de evaluación: clasificación de sentimiento de reseñas
RESEÑAS = [
    {"texto": "El producto es excelente, superó todas mis expectativas", "etiqueta": "positivo"},
    {"texto": "Muy decepcionante, no funciona como se describe", "etiqueta": "negativo"},
    {"texto": "El envío fue rápido pero el producto es mediocre", "etiqueta": "mixto"},
    {"texto": "Increíble relación calidad-precio, lo recomiendo totalmente", "etiqueta": "positivo"},
    {"texto": "Llegó roto y el servicio al cliente no respondió", "etiqueta": "negativo"},
    {"texto": "Cumple su función básica sin más", "etiqueta": "neutro"},
]

# Definir variantes de prompt a comparar
PROMPTS = {
    "v1_basico": """Clasifica el sentimiento de esta reseña: {texto}
Responde con una sola palabra: positivo, negativo, neutro o mixto.""",

    "v2_con_ejemplos": """Clasifica el sentimiento de reseñas de productos.

Ejemplos:
- "Perfecto, muy contento" → positivo
- "Una basura, no lo compres" → negativo
- "Bueno pero caro" → mixto
- "Hace lo que promete" → neutro

Reseña: {texto}
Sentimiento (una palabra):""",

    "v3_cot": """Analiza el sentimiento de esta reseña paso a paso.

Reseña: {texto}

1. Identifica palabras clave positivas y negativas
2. Considera el tono general
3. Clasifica como: positivo, negativo, neutro o mixto

Responde SOLO con JSON: {{"clasificacion": "<sentimiento>", "confianza": 0-1}}"""
}

def evaluar_prompt(nombre_prompt: str, template: str, reseñas: list) -> dict:
    """Evalúa una versión de prompt y registra en MLflow."""
    aciertos = 0
    latencias = []
    tokens_totales = 0
    resultados = []

    for reseña in reseñas:
        prompt = template.format(texto=reseña["texto"])
        inicio = time.perf_counter()
        resp = client.messages.create(
            model=MODELO, max_tokens=100,
            messages=[{"role": "user", "content": prompt}]
        )
        latencia = time.perf_counter() - inicio
        latencias.append(latencia)
        tokens_totales += resp.usage.input_tokens + resp.usage.output_tokens

        texto_resp = resp.content[0].text.strip().lower()
        if "json" in nombre_prompt or "{" in texto_resp:
            try:
                datos = json.loads(texto_resp.split("```")[-1] if "```" in texto_resp else texto_resp)
                clasificacion = datos.get("clasificacion", "").lower()
            except:
                clasificacion = texto_resp
        else:
            clasificacion = texto_resp.split()[0].strip(".,!")

        correcto = reseña["etiqueta"] in clasificacion or clasificacion in reseña["etiqueta"]
        if correcto:
            aciertos += 1
        resultados.append({"texto": reseña["texto"][:40], "esperado": reseña["etiqueta"],
                           "predicho": clasificacion, "correcto": correcto})

    precision = aciertos / len(reseñas)
    latencia_media = sum(latencias) / len(latencias)

    return {
        "precision": precision,
        "latencia_media": latencia_media,
        "tokens_totales": tokens_totales,
        "resultados": resultados
    }

print("Evaluando versiones de prompt...")
metricas_por_version = {}

for nombre, template in PROMPTS.items():
    print(f"  Evaluando {nombre}...")
    metricas = evaluar_prompt(nombre, template, RESEÑAS)
    metricas_por_version[nombre] = metricas

print("\n=== RESULTADOS ===")
for nombre, m in metricas_por_version.items():
    print(f"  {nombre}: precisión={m['precision']:.1%} latencia={m['latencia_media']:.2f}s tokens={m['tokens_totales']}")

## 4. Registrar runs en MLflow

In [ ]:
# Registrar cada versión como un run en MLflow
run_ids = {}

for nombre, template in PROMPTS.items():
    metricas = metricas_por_version[nombre]

    with mlflow.start_run(run_name=nombre) as run:
        # Parámetros del experimento
        mlflow.log_params({
            "modelo": MODELO,
            "version_prompt": nombre,
            "num_ejemplos_eval": len(RESEÑAS),
            "longitud_prompt": len(template),
            "tiene_few_shot": "ejemplos" in nombre,
            "tiene_cot": "cot" in nombre,
        })

        # Métricas de rendimiento
        mlflow.log_metrics({
            "precision": metricas["precision"],
            "latencia_media_s": metricas["latencia_media"],
            "tokens_totales": metricas["tokens_totales"],
            "coste_estimado_usd": metricas["tokens_totales"] * 0.00000025,  # Haiku pricing
        })

        # Guardar prompt como artefacto
        with open(f"prompt_{nombre}.txt", "w") as f:
            f.write(template)
        mlflow.log_artifact(f"prompt_{nombre}.txt", "prompts")

        # Guardar resultados detallados
        df_resultados = pd.DataFrame(metricas["resultados"])
        df_resultados.to_csv(f"resultados_{nombre}.csv", index=False)
        mlflow.log_artifact(f"resultados_{nombre}.csv", "resultados")

        run_ids[nombre] = run.info.run_id
        print(f"✓ Run registrado: {nombre} (ID: {run.info.run_id[:8]}...)")

print("\nTodos los runs registrados en MLflow.")
print("Para ver la UI: mlflow ui --port 5000")

## 5. Comparar versiones y seleccionar la mejor

In [ ]:
# Comparativa entre versiones
filas = []
for nombre, m in metricas_por_version.items():
    filas.append({
        "Version": nombre,
        "Precision": f"{m['precision']:.1%}",
        "Latencia (s)": f"{m['latencia_media']:.2f}",
        "Tokens/eval": m["tokens_totales"] // len(RESEÑAS),
        "Coste/1K evals ($)": f"{m['tokens_totales'] * 0.00000025 * 1000 / len(RESEÑAS):.4f}",
    })

df_comparativa = pd.DataFrame(filas)
print("=== COMPARATIVA DE VERSIONES ===")
print(df_comparativa.to_string(index=False))

# Seleccionar mejor versión (mejor precisión con tokens razonables)
mejor_nombre = max(metricas_por_version.items(),
                   key=lambda x: x[1]["precision"] - x[1]["tokens_totales"] * 0.00001)[0]
print(f"\n✓ Mejor versión seleccionada: {mejor_nombre}")

# Simular registro en Model Registry
prompt_produccion = {
    "version": mejor_nombre,
    "template": PROMPTS[mejor_nombre],
    "metricas": {
        "precision": metricas_por_version[mejor_nombre]["precision"],
        "latencia_media": metricas_por_version[mejor_nombre]["latencia_media"],
    },
    "estado": "produccion",
    "fecha_promocion": datetime.now().isoformat(),
    "mlflow_run_id": run_ids[mejor_nombre],
}

with open("prompt_produccion.json", "w", encoding="utf-8") as f:
    json.dump(prompt_produccion, f, ensure_ascii=False, indent=2)

print(f"\n=== CONFIGURACIÓN DE PRODUCCIÓN ===")
print(json.dumps(prompt_produccion, ensure_ascii=False, indent=2))
print("\n✓ prompt_produccion.json guardado — usa este archivo para cargar el prompt en producción")